# KoELECTRA 분류기 학습 (Google Colab)

lotte-insight 프로젝트 — `training/train_classifier.py` Colab 실행용 노트북

**학습 대상**: 멀티라벨 뉴스 기사 분류기 (`monologg/koelectra-small-v3-discriminator` fine-tuning)

**사전 준비 (Google Drive 업로드)**
```
MyDrive/lotte-insight-data/
  labeled_titles.csv           ← training/data/labeled_titles.csv
  labeled_players.csv          ← training/data/labeled_players.csv (또는 resolved 버전)
```

In [ ]:
# 1. GPU 확인
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA:', torch.version.cuda)
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('[WARN] GPU not available — training will be very slow on CPU')

In [ ]:
# 2. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. 레포 클론 및 의존성 설치
# GitHub URL을 본인 레포로 교체하시오
GITHUB_REPO_URL = 'https://github.com/YOUR_USERNAME/lotte-insight.git'

!git clone {GITHUB_REPO_URL} /content/lotte-insight
%cd /content/lotte-insight/training
!pip install -q transformers==4.40.2 scikit-learn pandas torch sentencepiece

In [ ]:
# 4. 데이터 파일을 Google Drive에서 복사
import shutil, os

DRIVE_DATA_DIR = '/content/drive/MyDrive/lotte-insight-data'
LOCAL_DATA_DIR = '/content/lotte-insight/training/data'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

files_to_copy = [
    ('labeled_titles.csv',  'labeled_titles.csv'),
    ('labeled_players.csv', 'labeled_players.csv'),
]

for src_name, dst_name in files_to_copy:
    src = f'{DRIVE_DATA_DIR}/{src_name}'
    dst = f'{LOCAL_DATA_DIR}/{dst_name}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Copied: {src_name} -> {dst_name}')
    else:
        print(f'[WARN] Not found in Drive: {src}')

In [ ]:
# 5. 데이터 행 수 및 라벨 분포 확인
import pandas as pd

frames = []
for fname in ['labeled_titles.csv', 'labeled_players.csv']:
    path = f'{LOCAL_DATA_DIR}/{fname}'
    if not os.path.exists(path):
        print(f'[MISSING] {fname}')
        continue
    df = pd.read_csv(path, encoding='utf-8-sig')
    lotte = df[df['is_lotte_related'].astype(str).str.lower() == 'true']
    with_summary = (
        lotte['event_summary'].fillna('').astype(str).str.strip().ne('').sum()
        if 'event_summary' in df.columns else 'N/A'
    )
    print(f'{fname}: total={len(df)}, lotte={len(lotte)}, with_summary={with_summary}')
    frames.append(lotte)

if frames:
    combined = pd.concat(frames, ignore_index=True).drop_duplicates(subset=['title'])
    print(f'\n학습에 사용될 고유 행 수: {len(combined)}')
    if 'primary_label' in combined.columns:
        print('\n라벨 분포:')
        print(combined['primary_label'].value_counts().to_string())

In [ ]:
# 6. 학습 실행
# 기본값: epochs=5, lr=5e-5, batch=16
# T4 GPU 기준 약 5~10분 소요
!python train_classifier.py \
    --epochs 5 \
    --lr 5e-5 \
    --batch 16

In [ ]:
# 7. 학습된 모델을 Google Drive에 저장
DRIVE_MODEL_DIR = '/content/drive/MyDrive/lotte-insight-data/models/classifier_koelectra'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

LOCAL_MODEL_DIR = '/content/lotte-insight/training/models/classifier_koelectra'
if os.path.exists(LOCAL_MODEL_DIR):
    shutil.copytree(LOCAL_MODEL_DIR, DRIVE_MODEL_DIR, dirs_exist_ok=True)
    print(f'Model saved to Drive: {DRIVE_MODEL_DIR}')
    print('Files:', os.listdir(DRIVE_MODEL_DIR))
else:
    print('[ERROR] Model directory not found — training may have failed')

In [ ]:
# 8. (선택) 평가만 실행 — 이미 학습된 모델이 있을 때
# !python train_classifier.py --eval-only

In [ ]:
# 9. (선택) 추론 테스트 — 저장된 모델 동작 확인
import json, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = '/content/lotte-insight/training/models/classifier_koelectra'

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

with open(f'{MODEL_PATH}/label_encoder.json', encoding='utf-8') as f:
    labels = json.load(f)

thresholds_path = f'{MODEL_PATH}/label_thresholds.json'
if os.path.exists(thresholds_path):
    with open(thresholds_path, encoding='utf-8') as f:
        thresholds = json.load(f)
else:
    thresholds = {label: 0.5 for label in labels}

def predict(title: str, auxiliary: str = '') -> list[str]:
    enc = tokenizer(title, auxiliary, truncation='only_second', padding='max_length',
                    max_length=128, return_tensors='pt')
    with torch.no_grad():
        logits = model(**enc).logits[0]
    probs = torch.sigmoid(logits)
    result = [labels[i] for i, p in enumerate(probs) if p >= thresholds[labels[i]]]
    if not result:
        result = ['ETC']
    return result

# 테스트 샘플
tests = [
    '롯데 자이언츠 나균안, 오른쪽 팔꿈치 부상으로 엔트리 말소',
    '롯데 자이언츠, 삼성 라이온즈 꺾고 3연승 달성',
    '롯데 자이언츠 이대호 "올 시즌 목표는 팀 우승"',
    '롯데 자이언츠, 외국인 투수 찰리 반스 영입 확정',
]

for title in tests:
    predicted = predict(title)
    print(f'  [{", ".join(predicted)}] {title}')